In [ ]:
import pandas as pd
import numpy as np
import re
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:

df = pd.read_csv("/content/mess_feedback_forms.csv")

FileNotFoundError: [Errno 2] No such file or directory: '/content/updated_mess_feedback_forms.csv'

In [ ]:
df.columns = df.columns.str.strip()

In [ ]:
df = df.rename(columns={
    'What was the main dish today in breakfast?': 'breakfast_menu',
    'How was the portion size?': 'breakfast_portion',
    'Did you leave food?': 'breakfast_left',
    'If yes, how much?': 'breakfast_waste_amount',
    'Why did you leave?': 'breakfast_reason',

    'What was the main dish today in lunch?': 'lunch_menu',
    'How was the portion size? .1': 'lunch_portion',
    'Did you leave food? .1': 'lunch_left',
    'If yes, how much? .1': 'lunch_waste_amount',
    'Why did you leave?.1': 'lunch_reason',

    'What was the main dish today in dinner?': 'dinner_menu',
    'How was the portion size? .2': 'dinner_portion',
    'Did you leave food? .2': 'dinner_left',
    'If yes, how much? .2': 'dinner_waste_amount',
    'Why did you leave?.2': 'dinner_reason',

    'Describe your experience': 'feedback',
    'Mess hygiene': 'hygiene',
    'Overall satisfaction of food': 'rating'
})

In [ ]:
df.columns = df.columns.str.strip()

In [ ]:
df = df.dropna(subset=['feedback'])

df['rating'] = pd.to_numeric(df['rating'], errors='coerce')
df['rating'] = df['rating'].fillna(df['rating'].median())

In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df['clean_feedback'] = df['feedback'].apply(clean_text)

In [ ]:
urdu_map = {
    "acha": "good",
    "bura": "bad",
    "bekaar": "bad",
    "zabardast": "excellent",
    "thanda": "cold",
    "garam": "hot"
}

def normalize_urdu(text):
    for k, v in urdu_map.items():
        text = text.replace(k, v)
    return text

df['clean_feedback'] = df['clean_feedback'].apply(normalize_urdu)

In [ ]:
yes_no_map = {'Yes': 1, 'No': 0}

df['breakfast_left'] = df['breakfast_left'].map(yes_no_map)
df['lunch_left'] = df['lunch_left'].map(yes_no_map)
df['dinner_left'] = df['dinner_left'].map(yes_no_map)


In [ ]:
portion_map = {
    'Too much': 2,
    'Enough': 1,
    'Less': 0
}

df['breakfast_portion'] = df['breakfast_portion'].map(portion_map)
df['lunch_portion'] = df['lunch_portion'].map(portion_map)
df['dinner_portion'] = df['dinner_portion'].map(portion_map)

In [ ]:
waste_map = {
    'Little': 1,
    'Moderate': 2,
    'A lot': 3
}

df['breakfast_waste_amount'] = df['breakfast_waste_amount'].map(waste_map)
df['lunch_waste_amount'] = df['lunch_waste_amount'].map(waste_map)
df['dinner_waste_amount'] = df['dinner_waste_amount'].map(waste_map)

In [ ]:
df['total_waste'] = (
    df['breakfast_left'].fillna(0) +
    df['lunch_left'].fillna(0) +
    df['dinner_left'].fillna(0)
)


In [ ]:
df['feedback_length'] = df['clean_feedback'].apply(len)
df['word_count'] = df['clean_feedback'].apply(lambda x: len(x.split()))

df['avg_portion'] = df[['breakfast_portion','lunch_portion','dinner_portion']].mean(axis=1)
df['avg_waste'] = df[['breakfast_waste_amount','lunch_waste_amount','dinner_waste_amount']].mean(axis=1)

In [ ]:
vectorizer = TfidfVectorizer(ngram_range=(1,2), max_features=500)
X_text = vectorizer.fit_transform(df['clean_feedback'])

In [ ]:
print(df.head())
print(df.isnull().sum())
print(df.describe())

In [ ]:
# WASTELESS DATA CLEANING CODE

# IMPORT LIBRARIES

import pandas as pd
import numpy as np
import re

from sklearn.preprocessing import LabelEncoder



df = pd.read_csv("/content/mess_feedback_forms.csv")


# DISPLAY FIRST ROWS


print("\nFIRST 5 ROWS:\n")

print(df.head())


# CLEAN COLUMN NAMES


df.columns = df.columns.str.strip()

# CONVERT COLUMN NAMES TO LOWERCASE

df.columns = df.columns.str.lower()

# REPLACE SPACES WITH UNDERSCORES

df.columns = df.columns.str.replace(" ", "_")

print("\nCLEANED COLUMN NAMES:\n")

print(df.columns)


# REMOVE DUPLICATES

before_duplicates = len(df)

df = df.drop_duplicates()

after_duplicates = len(df)

print("\nDUPLICATES REMOVED:")

print(before_duplicates - after_duplicates)


# HANDLE MISSING VALUES


print("\nMISSING VALUES:\n")

print(df.isnull().sum())

# FILL TEXT COLUMNS WITH 'unknown'

text_columns = df.select_dtypes(include='object').columns

for col in text_columns:

    df[col] = df[col].fillna("unknown")

# FILL NUMERIC COLUMNS WITH MEAN

numeric_columns = df.select_dtypes(include=np.number).columns

for col in numeric_columns:

    df[col] = df[col].fillna(df[col].mean())

# CLEAN TEXT FUNCTION

def clean_text(text):

    text = str(text).lower()

    # REMOVE SPECIAL CHARACTERS

    text = re.sub(r"[^a-zA-Z\\s]", "", text)

    # REMOVE EXTRA SPACES

    text = re.sub(r"\\s+", " ", text)

    return text.strip()

for col in text_columns:

    df[col] = df[col].apply(clean_text)

print("\nTEXT CLEANING COMPLETED")

# ENCODE YES / NO VALUES

yes_no_map = {

    'yes': 1,

    'no': 0

}

for col in df.columns:

    if df[col].dtype == 'object':

        unique_values = df[col].unique()

        if 'yes' in unique_values or 'no' in unique_values:

            df[col] = df[col].map(yes_no_map)

print("\nYES/NO ENCODING COMPLETED")

# LABEL ENCODING FOR CATEGORICAL COLUMNS

encoder = LabelEncoder()

for col in text_columns:

    if df[col].dtype == 'object':

        df[col] = encoder.fit_transform(df[col])

print("\nLABEL ENCODING COMPLETED")


# REMOVE EXTRA WHITESPACES

df = df.applymap(

    lambda x:
    x.strip() if isinstance(x, str) else x

)


print("\nFINAL CLEANED DATASET:\n")

print(df.head())

# DATASET INFO

print("\nDATASET INFO:\n")

print(df.info())
df.to_csv(

    "/content/cleaned_final_balanced_400_forms.csv",

    index=False

)

print("\nCLEANED CSV FILE SAVED SUCCESSFULLY")